In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
import os
import qnas_config as cfg
from util import check_files, load_yaml
from cnn import input
from cnn.input import GenericDataLoader
from cnn.train_detailed import train_and_eval

In [3]:
phase = 'retrain'
experiment_path = os.path.join("retrain_experiments", "exp15_adamw_repeat_1")
config_file = 'config_files/config10.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'retrain_folder': 'retrai_0',
    'data_path': 'pathmnist_data',
    'dataset': 'pathmnist',
    'log_level': 'INFO',
    'max_epochs': 300,
    'epochs_to_eval': 10,
    'batch_size': 256,
    'eval_batch_size': 1000,
    'limit_data': False,
    'num_workers': 4,
    'device': 'cuda:0',
    'lr_scheduler': 'cosine',
}

In [4]:
check_files(args['experiment_path'])
config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict

In [5]:
config.load_evolved_data(experiment_path=experiment_path)
params = config.train_spec
params

{'available_gpus': [0, 2],
 'batch_size': 256,
 'data_augmentation': True,
 'data_path': 'pathmnist_data',
 'dataset': 'pathmnist',
 'decay': 0.9,
 'device': 'cuda:0',
 'epochs_to_eval': 10,
 'eval_batch_size': 1000,
 'experiment_path': 'retrain_experiments/exp15_adamw_repeat_1/retrai_0',
 'learning_rate': 0.001,
 'limit_data': False,
 'limit_data_value': 10000,
 'log_level': 'INFO',
 'max_epochs': 300,
 'mixed_precision': True,
 'momentum': 0.0,
 'num_workers': 4,
 'optimizer': 'AdamW',
 'phase': 'retrain',
 'save_checkpoints_epochs': 10,
 'save_summary_epochs': 0.25,
 'subtract_mean': True,
 'threads': 0,
 'weight_decay': 0.0001,
 'config_file': 'config_files/config10.txt',
 'retrain_folder': 'retrai_0',
 'lr_scheduler': 'cosine'}

In [6]:
evolved_params = config.evolved_params
evolved_params['net']

['no_op',
 'cbamconv_3_1_32',
 'no_op',
 'cbamconv_1_1_128',
 'conv_3_1_256',
 'conv_5_1_128',
 'no_op',
 'no_op',
 'cbamconv_5_1_32',
 'max_pool_2_2',
 'avg_pool_2_2',
 'no_op',
 'cbamconv_1_1_128',
 'conv_3_1_256',
 'avg_pool_2_2',
 'no_op',
 'cbamconv_5_1_64',
 'max_pool_2_2',
 'conv_5_1_32',
 'avg_pool_2_2']

In [7]:
data_loader = GenericDataLoader(params=params)

In [13]:
train_loader, val_loader = data_loader.get_loader(pin_memory_device='cuda:0')
test_loader = data_loader.get_loader(for_train=False, pin_memory_device='cuda:0')

In [14]:
# Initialize a dictionary to count images per class
class_counts = {}  # Create an empty dictionary

# Iterate through the DataLoader to count images per class
loaders = [train_loader, val_loader, test_loader]
for loader in loaders:
    for batch in loader:
        _, targets = batch
        for label in targets:
            label = label.item()  # Convert the tensor label to an integer
            if label not in class_counts:
                class_counts[label] = 0  # Initialize the count to 0 if it doesn't exist
            class_counts[label] += 1
    print("-" * 50)
    # Print the counts for each class
    for class_label, count in class_counts.items():
        print(f"Class {class_label}: {count} images")
    class_counts = {}  # Reset the dictionary

--------------------------------------------------
Class 0: 10048 images
Class 2: 10059 images
Class 7: 9989 images
Class 6: 10019 images
Class 5: 10055 images
Class 3: 10023 images
Class 4: 9965 images
Class 8: 9943 images
Class 1: 9895 images
--------------------------------------------------
Class 5: 1354 images
Class 0: 1041 images
Class 1: 1057 images
Class 7: 1045 images
Class 4: 890 images
Class 3: 1156 images
Class 8: 1432 images
Class 6: 877 images
Class 2: 1152 images
--------------------------------------------------
Class 8: 1233 images
Class 4: 1035 images
Class 0: 1338 images
Class 6: 741 images
Class 5: 592 images
Class 2: 339 images
Class 1: 847 images
Class 7: 421 images
Class 3: 634 images


In [ ]:
import torch
import numpy as np

In [ ]:
class UnNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        """
        Args:
            tensor (Tensor): Tensor image of size (C, H, W) to be normalized.
        Returns:
            Tensor: Normalized image.
        """
        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m)
            # The normalize code -> t.sub_(m).div_(s)
        return tensor
unorm = UnNormalize(mean=(0.74053, 0.5329, 0.70583), std=(0.12368, 0.17676, 0.1244))

In [ ]:
# plot some images of the pytorch train_loader
import matplotlib.pyplot as plt

#cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship','truck']
figure = plt.figure(figsize=(5, 5))
cols, rows = 5, 3
train_imgs, train_labels = next(iter(val_loader))
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_imgs), size=(1,)).item()
    img, label = train_imgs[sample_idx], train_labels[sample_idx]
    img = unorm(img)
    npimg = img.numpy()
    figure.add_subplot(rows, cols, i)
    #plt.title(cifar10_classes[label])
    plt.axis("off")
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
plt.autoscale(enable=True, axis='both', tight=True)
plt.show()

In [10]:
dataset_info = load_yaml(os.path.join(params['data_path'], 'data_info_copy.txt'))
type(dataset_info['class_weights'])



dict

In [12]:
import torch
class_weights_normalized = dataset_info['class_weights']
alpha = torch.tensor([class_weights_normalized[label] for label in sorted(class_weights_normalized.keys())])
alpha

tensor([0.1158, 0.1140, 0.1046, 0.1042, 0.1354, 0.0890, 0.1375, 0.1153, 0.0841])